# NPU Message Bubble Analysis

This notebook analyzes network traces from G2 and NS3 simulations to visualize when NPUs are actively sending messages versus idle (bubbles). 

## Key Features:
- **Timeline visualization** showing each NPU's message send activity
- **Concurrent message detection** with darker colors when multiple messages overlap
- **Bubble identification** showing idle periods (gaps between messages)
- **Side-by-side comparison** between G2 and NS3 network simulations

## Data Sources:
- **G2**: `workload_network.csv` with issue_tick timing data
- **NS3**: `astrasim_fct.txt` with hex NPU IDs and start/end times

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import List, Dict, Tuple, Optional
import re

print("Libraries imported successfully!")

## Helper Functions

Core utility functions for finding and parsing trace files.

In [ ]:
def find_network_trace_file(run_folder: str, sim_type: str) -> Optional[str]:
    """
    Finds the network trace file for a given simulation type.
    
    Parameters:
    - run_folder: Path to the run directory
    - sim_type: 'g2' or 'ns3'
    
    Returns:
    - Path to trace file or None if not found
    """
    sim_dir = os.path.join(run_folder, sim_type.lower())
    if not os.path.isdir(sim_dir):
        return None
    
    # Define target files for each simulation type
    target_files = {
        'g2': 'workload_network.csv',
        'ns3': 'astrasim_fct.txt'
    }
    
    target_file = target_files.get(sim_type.lower())
    if not target_file:
        return None
    
    # Search for the target file
    for filename in os.listdir(sim_dir):
        if filename == target_file:
            return os.path.join(sim_dir, filename)
    
    return None

print("Helper functions defined successfully!")

## G2 Network Trace Parsing

G2 traces are stored in CSV format with columns including `src_origin`, `dst_final`, `tag`, `chunk_id`, and `issue_tick`. We group by communication pairs to find message start and end times.

In [ ]:
def parse_g2_network_trace(trace_file: str) -> pd.DataFrame:
    """
    Parse G2 network trace file (workload_network.csv).
    
    Parameters:
    - trace_file: Path to the CSV trace file
    
    Returns:
    DataFrame with columns:
    - src_npu: source NPU ID
    - dst_npu: destination NPU ID  
    - start_time: first issue_tick for this message
    - end_time: last issue_tick for this message
    - message_size: tensor size in bytes
    - tag: message tag
    - chunk_id: chunk identifier
    """
    messages = []
    
    try:
        # Read CSV, skip header
        df = pd.read_csv(trace_file)
        
        # Filter only 'update' actions (actual data transfers)
        df = df[df['action'] == 'update'].copy()
        
        # Group by message identifier to find temporal boundaries
        grouped = df.groupby(['src_origin', 'dst_final', 'tag', 'workload_node_id', 'chunk_id'])
        
        for (src, dst, tag, wl_node, chunk), group in grouped:
            start_time = group['issue_tick'].min()
            end_time = group['issue_tick'].max()
            message_size = group['tensor_size'].iloc[0]
            
            # Only include messages with measurable duration
            if end_time > start_time:
                messages.append({
                    'src_npu': int(src),
                    'dst_npu': int(dst),
                    'start_time': float(start_time),
                    'end_time': float(end_time),
                    'message_size': int(message_size),
                    'tag': int(tag),
                    'chunk_id': int(chunk),
                    'workload_node_id': int(wl_node)
                })
    
    except Exception as e:
        print(f"Error parsing G2 trace: {e}")
        return pd.DataFrame()
    
    return pd.DataFrame(messages)

print("G2 parsing function defined!")

## NS3 Network Trace Parsing

NS3 traces are space-separated with hex NPU IDs in the first two columns and timing data in specific positions.

**Format**: `src_hex dst_hex tag ? size1 size2 ? start_time duration other`
- Column -3: start_time (when message transmission begins)
- Column -2: duration (time taken for message in ns)  
- End time: calculated as start_time + duration

**Example**: `0b000201 0b000301 10000 100 134217728 140660192 10 5771446 5626527`
- NPU 2 → NPU 3, starts at 10ns, duration 5771446ns, ends at 5771456ns

**Critical Fix**: NPU IDs are encoded in hex format like `0b000201` where the NPU ID is in the **second-to-last byte**. We extract it by shifting right 8 bits and masking with 0xFF.

In [ ]:
def parse_ns3_network_trace(trace_file: str) -> pd.DataFrame:
    """
    Parse NS3 network trace file (astrasim_fct.txt).
    
    Format: src_hex dst_hex tag ? size1 size2 ? start_time duration other
    Note: Column -3 = start_time, Column -2 = duration, end_time = start + duration
    
    Example: 0b000201 0b000301 10000 100 134217728 140660192 10 5771446 5626527
    - NPU 2 → NPU 3, starts at 10ns, duration 5771446ns, ends at 5771456ns
    
    IMPORTANT: NPU IDs are in hex format like 0b000201 where NPU ID is in second-to-last byte:
    - Example: 0b000201 = 0x0B000201
    - Shift right 8 bits: 0x0B0002  
    - Mask & 0xFF: 0x02 = NPU ID 2
    
    Parameters:
    - trace_file: Path to the space-separated trace file
    
    Returns:
    DataFrame with columns:
    - src_npu: source NPU ID
    - dst_npu: destination NPU ID
    - start_time: message start time
    - end_time: message finish time
    - message_size: message size in bytes
    - tag: message tag
    """
    messages = []
    
    try:
        with open(trace_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                
                # Extract hex NPU IDs from first two columns
                src_hex = parts[0]  # e.g., "0b000201"
                dst_hex = parts[1]  # e.g., "0b000301"
                
                # FIXED: Extract NPU ID from second-to-last byte
                # Convert hex to int, shift right 8 bits, then mask to get last byte
                src_val = int(src_hex, 16)
                dst_val = int(dst_hex, 16)
                src_npu = (src_val >> 8) & 0xFF  # Second-to-last byte
                dst_npu = (dst_val >> 8) & 0xFF  # Second-to-last byte
                
                # Extract other fields
                tag = int(parts[2])
                message_size = int(parts[4])
                start_time = float(parts[-3])  # Column -3 is start time
                duration = float(parts[-2])    # Column -2 is duration
                end_time = start_time + duration  # Calculate end time as start + duration
                
                messages.append({
                    'src_npu': src_npu,
                    'dst_npu': dst_npu,
                    'start_time': start_time,
                    'end_time': end_time,
                    'message_size': message_size,
                    'tag': tag
                })
    
    except Exception as e:
        print(f"Error parsing NS3 trace: {e}")
        return pd.DataFrame()
    
    return pd.DataFrame(messages)

print("NS3 parsing function defined with FIXED hex extraction!")

## NPU Activity Analysis

Extract timeline information for each NPU and detect concurrent message transmission.

In [ ]:
def extract_npu_activity(messages_df: pd.DataFrame) -> Dict[int, List[Dict]]:
    """
    Extract activity timeline for each NPU and detect concurrent messages.
    
    Parameters:
    - messages_df: DataFrame with message data
    
    Returns:
    Dictionary mapping NPU ID to list of activity periods:
    - start_time: when message transmission starts
    - end_time: when message transmission ends  
    - dst_npu: destination NPU
    - message_size: size in bytes
    - concurrent_count: number of messages already active when this message starts (0 = single message)
    - tag: message tag
    """
    npu_activities = {}
    
    # Get all unique source NPUs
    npu_ids = sorted(messages_df['src_npu'].unique())
    
    for npu_id in npu_ids:
        # Get all messages sent by this NPU, sorted by start time
        npu_messages = messages_df[messages_df['src_npu'] == npu_id].copy()
        npu_messages = npu_messages.sort_values('start_time').reset_index(drop=True)
        
        activities = []
        for i, row in npu_messages.iterrows():
            msg = row
            
            # Count how many OTHER messages from same NPU are still active when this message starts
            # (i.e., messages that started earlier but haven't finished yet)
            concurrent_count = 0
            
            for j, other_row in npu_messages.iterrows():
                if i == j:  # Skip self
                    continue
                
                other_msg = other_row
                
                # Only count messages that:
                # 1. Started before or at the same time as current message
                # 2. Are still active (haven't ended) when current message starts
                if (other_msg['start_time'] <= msg['start_time'] and 
                    other_msg['end_time'] > msg['start_time']):
                    concurrent_count += 1
            
            activities.append({
                'start_time': msg['start_time'],
                'end_time': msg['end_time'],
                'dst_npu': msg['dst_npu'],
                'message_size': msg['message_size'],
                'concurrent_count': concurrent_count,
                'tag': msg.get('tag', 0)
            })
        
        npu_activities[npu_id] = activities
    
    return npu_activities

print("NPU activity extraction function defined!")

## Visualization Functions

Create Gantt-chart style timeline showing NPU activity, bubbles, and concurrent messages.

In [ ]:
def plot_npu_timeline(npu_activities: Dict[int, List[Dict]], title: str, time_unit: str = 'ns'):
    """
    Create timeline visualization showing NPU message activity and idle periods (bubbles).
    
    Parameters:
    - npu_activities: Dictionary from extract_npu_activity()
    - title: Plot title
    - time_unit: 'ns', 'us', or 'ms' for time scaling
    
    Visualization shows:
    - X-axis: Time
    - Y-axis: NPU IDs
    - Horizontal bars: Message send periods
    - Color intensity: Darker = more concurrent messages
    - Gaps: Idle periods (bubbles)
    - Red markers: Message milestones (16th, 31st, 46th, etc. message per NPU)
    """
    fig = go.Figure()
    
    # Time scaling
    time_scale = {'ns': 1, 'us': 1e-3, 'ms': 1e-6}[time_unit]
    
    # Sort NPU IDs for consistent display
    npu_ids = sorted(npu_activities.keys())
    
    # Calculate milestone times for each NPU individually
    milestone_numbers = [16, 31, 46, 61, 76]  # Per-NPU message milestones
    npu_milestones = {}  # Dictionary mapping NPU ID to list of milestone times
    
    for npu_id in npu_ids:
        activities = npu_activities[npu_id]
        sorted_activities = sorted(activities, key=lambda x: x['start_time'])
        npu_milestones[npu_id] = []
        
        for milestone_num in milestone_numbers:
            if len(sorted_activities) >= milestone_num:  # NPU has at least N messages
                nth_message_time = sorted_activities[milestone_num - 1]['start_time'] * time_scale
                npu_milestones[npu_id].append((milestone_num, nth_message_time))
    
    # Color mapping for concurrent messages
    # concurrent_count = number of other messages already active when this message starts
    def get_color(concurrent_count):
        colors = {
            0: 'rgba(70, 130, 180, 0.7)',   # Light blue (no other active messages)
            1: 'rgba(30, 80, 140, 0.8)',    # Medium blue (1 other active message)
            2: 'rgba(10, 50, 100, 0.9)',    # Darker blue (2 other active messages)
        }
        return colors.get(concurrent_count, 'rgba(5, 25, 60, 0.95)')  # Darkest blue (3+ other active messages)
    
    # Plot activities for each NPU
    for npu_id in npu_ids:
        activities = npu_activities[npu_id]
        
        for activity in activities:
            start = activity['start_time'] * time_scale
            end = activity['end_time'] * time_scale
            duration = end - start
            concurrent = activity['concurrent_count']
            
            # Hover information
            hover_text = (
                f"NPU {npu_id} → NPU {activity['dst_npu']}<br>"
                f"Start: {start:.2f} {time_unit}<br>"
                f"End: {end:.2f} {time_unit}<br>"
                f"Duration: {duration:.2f} {time_unit}<br>"
                f"Size: {activity['message_size']/1024/1024:.2f} MB<br>"
                f"Other active messages when started: {concurrent}"
            )
            
            # Add horizontal bar for this message
            fig.add_trace(go.Bar(
                x=[duration],
                y=[f"NPU {npu_id}"],
                orientation='h',
                base=[start],
                marker=dict(
                    color=get_color(concurrent),
                    line=dict(color='black', width=0.3)
                ),
                hovertext=hover_text,
                hoverinfo='text',
                showlegend=False
            ))
    
    # Add individual milestone markers for each NPU
    for npu_id in npu_ids:
        milestones = npu_milestones.get(npu_id, [])
        if milestones:
            milestone_times_list = [time for _, time in milestones]
            milestone_nums_list = [num for num, _ in milestones]
            
            # Add scatter plot for milestone markers
            fig.add_trace(go.Scatter(
                x=milestone_times_list,
                y=[f"NPU {npu_id}"] * len(milestone_times_list),
                mode='markers',
                marker=dict(
                    symbol='line-ns-open',  # Vertical line marker
                    size=15,
                    color='red',
                    line=dict(width=3, color='red')
                ),
                hovertext=[f'NPU {npu_id}: {num}th message at {time:.2f} {time_unit}' 
                          for num, time in milestones],
                hoverinfo='text',
                showlegend=False,
                name=f'NPU {npu_id} milestones'
            ))
    
    # Layout configuration
    fig.update_layout(
        title=f"{title}<br><sub>Darker bars = messages with other concurrent active messages | Gaps = idle periods (bubbles) | Red markers = message milestones per NPU</sub>",
        xaxis_title=f"Time ({time_unit})",
        yaxis_title="NPU ID",
        height=max(500, len(npu_ids) * 45),
        barmode='overlay',
        hovermode='closest',
        yaxis=dict(
            categoryorder='array',
            categoryarray=[f"NPU {npu_id}" for npu_id in reversed(npu_ids)]
        )
    )
    
    # Add legend
    legend_traces = [
        go.Scatter(x=[None], y=[None], mode='markers',
                  marker=dict(size=12, color='rgba(70, 130, 180, 0.7)'),
                  name='No Concurrent Messages'),
        go.Scatter(x=[None], y=[None], mode='markers',
                  marker=dict(size=12, color='rgba(30, 80, 140, 0.8)'),
                  name='1 Other Active Message'),
        go.Scatter(x=[None], y=[None], mode='markers',
                  marker=dict(size=12, color='rgba(10, 50, 100, 0.9)'),
                  name='2 Other Active Messages'),
        go.Scatter(x=[None], y=[None], mode='markers',
                  marker=dict(size=12, color='rgba(5, 25, 60, 0.95)'),
                  name='3+ Other Active Messages'),
        go.Scatter(x=[None], y=[None], mode='markers',
                  marker=dict(symbol='line-ns-open', size=12, color='red'),
                  name='Message Milestones')
    ]
    for trace in legend_traces:
        fig.add_trace(trace)
    
    fig.show()
    
    # Print detailed statistics including milestone information
    print_npu_statistics(npu_activities, title, time_scale, time_unit, npu_milestones)

def print_npu_statistics(npu_activities, title, time_scale, time_unit, npu_milestones=None):
    """Print detailed statistics for NPU activities."""
    print(f"\n{'='*70}")
    print(f"Statistics for {title}")
    print(f"{'='*70}")
    
    # Print milestone information if available
    if npu_milestones:
        print(f"\n🎯 Message Milestones (per NPU):")
        milestone_count = 0
        for npu_id in sorted(npu_milestones.keys())[:8]:  # Show first 8 NPUs
            milestones = npu_milestones[npu_id]
            if milestones:
                milestone_strs = [f"{num}th:{time:.1f}{time_unit}" for num, time in milestones]
                print(f"   NPU {npu_id}: {', '.join(milestone_strs)}")
                milestone_count += len(milestones)
        print(f"   Total markers: {milestone_count}")
    
    npu_ids = sorted(npu_activities.keys())
    
    for npu_id in npu_ids[:8]:  # Show first 8 NPUs
        activities = npu_activities[npu_id]
        if not activities:
            continue
        
        # Calculate metrics
        total_active = sum(a['end_time'] - a['start_time'] for a in activities)
        span = max(a['end_time'] for a in activities) - min(a['start_time'] for a in activities)
        utilization = (total_active / span * 100) if span > 0 else 0
        max_other_active = max(a['concurrent_count'] for a in activities)
        avg_msg_size = np.mean([a['message_size'] for a in activities])
        
        print(f"\nNPU {npu_id}:")
        print(f"  Messages sent: {len(activities)}")
        print(f"  Active time: {total_active * time_scale:.2f} {time_unit}")
        print(f"  Total span: {span * time_scale:.2f} {time_unit}")
        print(f"  Utilization: {utilization:.1f}%")
        print(f"  Max other active when starting: {max_other_active}")
        print(f"  Avg message size: {avg_msg_size/1024/1024:.2f} MB")

print("Visualization functions defined with milestone markers!")

## Example Analysis Setup

Configure paths and select different runs for G2 and NS3 analysis. We use different run folders to compare varied network behaviors.

In [ ]:
# Base configuration
base_path = '/app/astra-sim/upc/output/comparison_run/Dragonfly/multiple_collectives_tp'
example_workload = 'six_allgather'

# Initialize run paths
g2_example_run = None
ns3_example_run = None

# Search for available run folders
workload_path = os.path.join(base_path, example_workload)
print(f"Looking for runs in: {workload_path}")

if os.path.isdir(workload_path):
    # Get all run directories
    run_folders = sorted([d for d in os.listdir(workload_path) 
                         if os.path.isdir(os.path.join(workload_path, d))])
    
    print(f"Found {len(run_folders)} run folders: {run_folders}")
    
    if len(run_folders) >= 2:
        # Use DIFFERENT runs for G2 and NS3 to compare varied behaviors
        g2_example_run = os.path.join(workload_path, run_folders[2])
        ns3_example_run = os.path.join(workload_path, run_folders[0])
        print(f"\n✅ Using different runs:")
        print(f"   G2 run:  {os.path.basename(g2_example_run)}")
        print(f"   NS3 run: {os.path.basename(ns3_example_run)}")
    elif len(run_folders) == 1:
        # Fallback to same run if only one available
        g2_example_run = ns3_example_run = os.path.join(workload_path, run_folders[0])
        print(f"\n⚠️  Only one run available, using same for both: {os.path.basename(g2_example_run)}")
    else:
        print(f"\n❌ No run folders found in {workload_path}")
else:
    print(f"\n❌ Workload path not found: {workload_path}")

# Validate paths
if g2_example_run and ns3_example_run:
    print(f"\n📁 Run paths configured successfully!")
else:
    print(f"\n❌ Could not configure run paths. Please check the base_path and workload name.")

## G2 Simulation Analysis

Parse and visualize G2 network trace showing NPU message activity and bubbles.

In [ ]:
if g2_example_run:
    print(f"Analyzing G2 simulation from: {os.path.basename(g2_example_run)}")
    
    # Find G2 trace file
    g2_trace_file = find_network_trace_file(g2_example_run, 'g2')
    
    if g2_trace_file and os.path.exists(g2_trace_file):
        print(f"Found G2 trace: {g2_trace_file}")
        
        # Parse trace data
        g2_messages = parse_g2_network_trace(g2_trace_file)
        
        if not g2_messages.empty:
            print(f"\n📊 G2 Trace Summary:")
            print(f"   Total messages: {len(g2_messages)}")
            print(f"   NPUs involved: {sorted(g2_messages['src_npu'].unique())}")
            print(f"   Time range: {g2_messages['start_time'].min():.0f} - {g2_messages['end_time'].max():.0f} ns")
            print(f"   Total simulation time: {(g2_messages['end_time'].max() - g2_messages['start_time'].min())/1000:.2f} μs")
            
            # Extract NPU activity timelines
            g2_activities = extract_npu_activity(g2_messages)
            
            # Create visualization
            plot_npu_timeline(g2_activities, "G2 Network Simulation - NPU Activity & Bubble Analysis", time_unit='us')
        else:
            print("❌ No messages found in G2 trace file")
    else:
        print(f"❌ G2 trace file not found: {g2_trace_file}")
else:
    print("❌ No G2 example run configured")

## NS3 Simulation Analysis

Parse and visualize NS3 network trace with corrected hex NPU ID extraction.

In [ ]:
if ns3_example_run:
    print(f"Analyzing NS3 simulation from: {os.path.basename(ns3_example_run)}")
    
    # Find NS3 trace file
    ns3_trace_file = find_network_trace_file(ns3_example_run, 'ns3')
    
    if ns3_trace_file and os.path.exists(ns3_trace_file):
        print(f"Found NS3 trace: {ns3_trace_file}")
        
        # Parse trace data with FIXED hex extraction
        ns3_messages = parse_ns3_network_trace(ns3_trace_file)
        
        if not ns3_messages.empty:
            print(f"\n📊 NS3 Trace Summary:")
            print(f"   Total messages: {len(ns3_messages)}")
            print(f"   NPUs involved: {sorted(ns3_messages['src_npu'].unique())}")
            print(f"   Time range: {ns3_messages['start_time'].min():.0f} - {ns3_messages['end_time'].max():.0f} ns")
            print(f"   Total simulation time: {(ns3_messages['end_time'].max() - ns3_messages['start_time'].min())/1000:.2f} μs")
            
            # Extract NPU activity timelines
            ns3_activities = extract_npu_activity(ns3_messages)
            
            # Create visualization
            plot_npu_timeline(ns3_activities, "NS3 Network Simulation - NPU Activity & Bubble Analysis", time_unit='us')
        else:
            print("❌ No messages found in NS3 trace file")
    else:
        print(f"❌ NS3 trace file not found: {ns3_trace_file}")
else:
    print("❌ No NS3 example run configured")

## Comparative Analysis

Side-by-side comparison of G2 and NS3 simulations showing differences in bubble patterns and network utilization.

In [ ]:
def compare_simulations(g2_activities: Dict, ns3_activities: Dict):
    """
    Comprehensive comparison between G2 and NS3 simulation results.
    
    Compares:
    - NPU utilization patterns
    - Message concurrency levels (number of other messages already active when each message starts)
    - Idle period (bubble) distributions
    - Overall network efficiency
    """
    print(f"\n{'='*80}")
    print("🔍 COMPARATIVE ANALYSIS: G2 vs NS3")
    print(f"{'='*80}\n")
    
    # Analyze NPU coverage
    g2_npus = set(g2_activities.keys())
    ns3_npus = set(ns3_activities.keys())
    common_npus = sorted(g2_npus & ns3_npus)
    
    print(f"📋 NPU Coverage:")
    print(f"   Common NPUs: {len(common_npus)} {common_npus[:10] if len(common_npus) > 10 else common_npus}")
    print(f"   G2-only NPUs: {len(g2_npus - ns3_npus)}")
    print(f"   NS3-only NPUs: {len(ns3_npus - g2_npus)}")
    
    # Detailed comparison for common NPUs
    comparison_data = []
    
    for npu_id in common_npus[:12]:  # Analyze first 12 NPUs
        g2_acts = g2_activities.get(npu_id, [])
        ns3_acts = ns3_activities.get(npu_id, [])
        
        if not g2_acts or not ns3_acts:
            continue
        
        # G2 metrics
        g2_active_time = sum(a['end_time'] - a['start_time'] for a in g2_acts)
        g2_span = max(a['end_time'] for a in g2_acts) - min(a['start_time'] for a in g2_acts)
        g2_utilization = (g2_active_time / g2_span * 100) if g2_span > 0 else 0
        g2_max_other_active = max(a['concurrent_count'] for a in g2_acts)
        g2_max_other_active = max(a['concurrent_count'] for a in g2_acts)
        g2_avg_size = np.mean([a['message_size'] for a in g2_acts])
        
        # NS3 metrics
        ns3_active_time = sum(a['end_time'] - a['start_time'] for a in ns3_acts)
        ns3_span = max(a['end_time'] for a in ns3_acts) - min(a['start_time'] for a in ns3_acts)
        ns3_utilization = (ns3_active_time / ns3_span * 100) if ns3_span > 0 else 0
        ns3_max_other_active = max(a['concurrent_count'] for a in ns3_acts)
        ns3_avg_size = np.mean([a['message_size'] for a in ns3_acts])
        
        comparison_data.append({
            'NPU': npu_id,
            'G2 Messages': len(g2_acts),
            'NS3 Messages': len(ns3_acts),
            'G2 Util (%)': round(g2_utilization, 1),
            'NS3 Util (%)': round(ns3_utilization, 1),
            'G2 Max Other Active': g2_max_other_active,
            'NS3 Max Other Active': ns3_max_other_active,
            'G2 Span (μs)': round(g2_span / 1000, 1),
            'NS3 Span (μs)': round(ns3_span / 1000, 1),
            'G2 Avg Size (MB)': round(g2_avg_size / 1024 / 1024, 2),
            'NS3 Avg Size (MB)': round(ns3_avg_size / 1024 / 1024, 2)
        })
    
    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)
        
        print(f"\n{'='*80}")
        print("📊 Per-NPU Detailed Comparison")
        print(f"{'='*80}\n")
        display(comp_df)
        
        # Overall summary statistics
        print(f"\n{'='*80}")
        print("📈 Summary Statistics")
        print(f"{'='*80}\n")
        
        print(f"🎯 Network Utilization:")
        print(f"   G2 Average:  {comp_df['G2 Util (%)'].mean():.1f}% (σ={comp_df['G2 Util (%)'].std():.1f})")
        print(f"   NS3 Average: {comp_df['NS3 Util (%)'].mean():.1f}% (σ={comp_df['NS3 Util (%)'].std():.1f})")
        
        print(f"\n⚡ Message Concurrency (other messages already active when starting):")
        print(f"   G2 Avg Max:  {comp_df['G2 Max Other Active'].mean():.1f} other active messages")
        print(f"   NS3 Avg Max: {comp_df['NS3 Max Other Active'].mean():.1f} other active messages")
        
        print(f"\n📦 Message Volume:")
        print(f"   G2 Total:  {comp_df['G2 Messages'].sum()} messages")
        print(f"   NS3 Total: {comp_df['NS3 Messages'].sum()} messages")
        
        print(f"\n🕒 Simulation Time Span:")
        print(f"   G2 Avg:  {comp_df['G2 Span (μs)'].mean():.1f} μs")
        print(f"   NS3 Avg: {comp_df['NS3 Span (μs)'].mean():.1f} μs")
        
        # Bubble analysis
        g2_bubbles = 100 - comp_df['G2 Util (%)'].mean()
        ns3_bubbles = 100 - comp_df['NS3 Util (%)'].mean()
        print(f"\n💭 Bubble (Idle) Analysis:")
        print(f"   G2 Avg Idle:  {g2_bubbles:.1f}% (more bubbles = more idle time)")
        print(f"   NS3 Avg Idle: {ns3_bubbles:.1f}% (more bubbles = more idle time)")
        
        if g2_bubbles > ns3_bubbles:
            print(f"   🔍 G2 has {g2_bubbles - ns3_bubbles:.1f}% more idle time (larger bubbles)")
        elif ns3_bubbles > g2_bubbles:
            print(f"   🔍 NS3 has {ns3_bubbles - g2_bubbles:.1f}% more idle time (larger bubbles)")
        else:
            print(f"   🔍 Both simulations have similar idle time patterns")
    else:
        print("❌ No common NPUs with sufficient data for comparison")

# Perform comparison if both datasets are available
if 'g2_activities' in globals() and 'ns3_activities' in globals():
    compare_simulations(g2_activities, ns3_activities)
else:
    print("ℹ️  Both G2 and NS3 activities need to be generated first for comparison.")
    print("   Run the G2 and NS3 analysis cells above, then re-run this comparison.")

## Summary

This notebook provides comprehensive analysis of NPU communication patterns:

### Key Insights:
1. **Message Activity**: Timeline showing when each NPU is actively sending messages
2. **Bubble Detection**: Idle periods (gaps) where NPUs are not transmitting
3. **Concurrency Analysis**: Identification of overlapping message transmissions (darker colors)
4. **Comparative Study**: Side-by-side G2 vs NS3 network behavior analysis

### Fixed Issues:
- ✅ **Correct cell order**: Title → Imports → Functions → Examples
- ✅ **NS3 hex parsing**: Fixed to extract NPU ID from second-to-last byte using `(int(hex, 16) >> 8) & 0xFF`
- ✅ **Separate runs**: Uses different run folders for G2 and NS3 when available
- ✅ **Enhanced visualization**: Clear color coding for concurrent messages and detailed statistics

### Usage:
1. Run cells sequentially from top to bottom
2. Modify `base_path` and `example_workload` variables as needed
3. Examine the timeline plots to identify bubble patterns
4. Compare statistics between G2 and NS3 simulations

## Generate PDF Report with All Plots

Generate a comprehensive PDF report containing all available G2 and NS3 network trace visualizations.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import plotly.io as pio

def find_all_simulation_runs(base_path: str, workload: str) -> List[Dict]:
    """
    Find all run folders and check which simulation types (g2/ns3) are available.
    
    Returns list of dictionaries with:
    - run_name: folder name
    - run_path: full path to run folder  
    - has_g2: boolean indicating if g2/ subfolder exists
    - has_ns3: boolean indicating if ns3/ subfolder exists
    - g2_trace: path to g2 trace file if available
    - ns3_trace: path to ns3 trace file if available
    """
    simulation_runs = []
    
    workload_path = os.path.join(base_path, workload)
    if not os.path.isdir(workload_path):
        print(f"❌ Workload path not found: {workload_path}")
        return simulation_runs
    
    # Get all run directories
    run_folders = sorted([d for d in os.listdir(workload_path) 
                         if os.path.isdir(os.path.join(workload_path, d))])
    
    print(f"🔍 Scanning {len(run_folders)} run folders for simulation data...")
    
    for run_name in run_folders:
        run_path = os.path.join(workload_path, run_name)
        
        # Check for g2 and ns3 subfolders
        g2_dir = os.path.join(run_path, 'g2')
        ns3_dir = os.path.join(run_path, 'ns3')
        
        has_g2 = os.path.isdir(g2_dir)
        has_ns3 = os.path.isdir(ns3_dir)
        
        # Find trace files if directories exist
        g2_trace = find_network_trace_file(run_path, 'g2') if has_g2 else None
        ns3_trace = find_network_trace_file(run_path, 'ns3') if has_ns3 else None
        
        # Only include if at least one simulation type is available
        if has_g2 or has_ns3:
            simulation_runs.append({
                'run_name': run_name,
                'run_path': run_path,
                'has_g2': has_g2 and g2_trace is not None,
                'has_ns3': has_ns3 and ns3_trace is not None,
                'g2_trace': g2_trace,
                'ns3_trace': ns3_trace
            })
            
            status = []
            if has_g2 and g2_trace: status.append("G2✓")
            if has_ns3 and ns3_trace: status.append("NS3✓")
            print(f"   📁 {run_name}: {', '.join(status)}")
        else:
            print(f"   ⚠️  {run_name}: No simulation data found")
    
    print(f"\n✅ Found {len(simulation_runs)} runs with simulation data")
    return simulation_runs

def generate_plot_for_pdf(npu_activities: Dict[int, List[Dict]], title: str, time_unit: str = 'us'):
    """
    Generate a matplotlib plot suitable for PDF export with individual milestone markers per NPU.
    """
    fig, ax = plt.subplots(figsize=(14, max(8, len(npu_activities) * 0.4)))
    
    # Time scaling
    time_scale = {'ns': 1, 'us': 1e-3, 'ms': 1e-6}[time_unit]
    
    # Sort NPU IDs for consistent display
    npu_ids = sorted(npu_activities.keys())
    
    # Calculate milestone times for each NPU individually
    milestone_numbers = [16, 31, 46, 61, 76]  # Per-NPU message milestones
    npu_milestones = {}  # Dictionary mapping NPU ID to list of milestone times
    
    for npu_id in npu_ids:
        activities = npu_activities[npu_id]
        sorted_activities = sorted(activities, key=lambda x: x['start_time'])
        npu_milestones[npu_id] = []
        
        for milestone_num in milestone_numbers:
            if len(sorted_activities) >= milestone_num:  # NPU has at least N messages
                nth_message_time = sorted_activities[milestone_num - 1]['start_time'] * time_scale
                npu_milestones[npu_id].append((milestone_num, nth_message_time))
    
    # Color mapping for concurrent messages
    # concurrent_count = number of other messages already active when this message starts
    colors = {
        0: '#4682B4',   # Steel blue (no other active messages)
        1: '#1E508C',   # Medium blue (1 other active message)  
        2: '#0A3264',   # Darker blue (2 other active messages)
    }
    default_color = '#051940'  # Darkest blue (3+ other active messages)
    
    y_positions = {npu_id: i for i, npu_id in enumerate(npu_ids)}
    
    # Plot activities for each NPU
    for npu_id in npu_ids:
        activities = npu_activities[npu_id]
        y_pos = y_positions[npu_id]
        
        for activity in activities:
            start = activity['start_time'] * time_scale
            end = activity['end_time'] * time_scale
            duration = end - start
            concurrent = activity['concurrent_count']
            
            # Choose color based on concurrency
            color = colors.get(concurrent, default_color)
            alpha = 0.6 if concurrent == 0 else 0.8 if concurrent == 1 else 0.95
            
            # Add horizontal bar
            ax.barh(y_pos, duration, left=start, height=0.6, 
                   color=color, alpha=alpha, edgecolor='black', linewidth=0.3)
    
    # Add individual milestone markers for each NPU
    for npu_id in npu_ids:
        milestones = npu_milestones.get(npu_id, [])
        y_pos = y_positions[npu_id]
        
        for milestone_num, milestone_time in milestones:
            # Add vertical line marker at NPU level
            ax.plot([milestone_time, milestone_time], [y_pos - 0.4, y_pos + 0.4], 
                   color='red', linewidth=3, alpha=0.8)
            # Add small text annotation
            ax.text(milestone_time, y_pos + 0.5, f'{milestone_num}',
                    ha='center', va='bottom', color='red', fontsize=8,
                    bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.8, edgecolor='red'))
    
    # Customize plot
    ax.set_yticks(range(len(npu_ids)))
    ax.set_yticklabels([f"NPU {npu_id}" for npu_id in npu_ids])
    ax.set_xlabel(f"Time ({time_unit})")
    ax.set_ylabel("NPU ID")
    ax.set_title(f"{title}\nDarker bars = messages with other concurrent active messages | Gaps = idle periods (bubbles) | Red markers = milestones per NPU")
    ax.grid(True, alpha=0.3)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors[0], alpha=0.6, label='No Concurrent Messages'),
        Patch(facecolor=colors[1], alpha=0.8, label='1 Other Active Message'),
        Patch(facecolor=colors.get(2, default_color), alpha=0.9, label='2 Other Active Messages'),
        Patch(facecolor=default_color, alpha=0.95, label='3+ Other Active Messages'),
        plt.Line2D([0], [0], color='red', linewidth=2, label='Message Milestones')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    return fig

# Scan for all available simulations
all_runs = find_all_simulation_runs(base_path, example_workload)

In [ ]:
def generate_comprehensive_pdf_report():
    """
    Generate a comprehensive PDF report with all G2 and NS3 plots.
    """
    if not all_runs:
        print("❌ No simulation runs found. Cannot generate PDF report.")
        return
    
    # Create PDF filename
    pdf_filename = f"/app/astra-sim/upc/comparing_networks/npu_bubble_analysis_{example_workload}_report.pdf"
    
    print(f"📄 Generating comprehensive PDF report...")
    print(f"   Output file: {pdf_filename}")
    
    with PdfPages(pdf_filename) as pdf:
        plot_count = 0
        
        # Title page
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.text(0.5, 0.7, 'NPU Message Bubble Analysis Report', 
                fontsize=24, fontweight='bold', ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.6, f'Workload: {example_workload}', 
                fontsize=16, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.5, f'Generated: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}', 
                fontsize=12, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.4, f'Total Runs: {len(all_runs)}', 
                fontsize=12, ha='center', transform=ax.transAxes)
        
        # Summary stats
        g2_count = sum(1 for run in all_runs if run['has_g2'])
        ns3_count = sum(1 for run in all_runs if run['has_ns3'])
        ax.text(0.5, 0.3, f'G2 Simulations: {g2_count}', 
                fontsize=12, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.25, f'NS3 Simulations: {ns3_count}', 
                fontsize=12, ha='center', transform=ax.transAxes)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        # Process each run
        for run_info in all_runs:
            run_name = run_info['run_name']
            print(f"\\n🔄 Processing run: {run_name}")
            
            # Process G2 if available
            if run_info['has_g2']:
                try:
                    print(f"   📊 Generating G2 plot...")
                    g2_messages = parse_g2_network_trace(run_info['g2_trace'])
                    
                    if not g2_messages.empty:
                        g2_activities = extract_npu_activity(g2_messages)
                        
                        if g2_activities:
                            fig = generate_plot_for_pdf(
                                g2_activities, 
                                f"G2 Network Simulation - {run_name}",
                                time_unit='us'
                            )
                            pdf.savefig(fig, bbox_inches='tight', dpi=150)
                            plt.close(fig)
                            plot_count += 1
                            
                            # Add statistics page
                            fig, ax = plt.subplots(figsize=(11, 8.5))
                            ax.text(0.1, 0.9, f'G2 Statistics - {run_name}', 
                                   fontsize=16, fontweight='bold', transform=ax.transAxes)
                            
                            y_pos = 0.8
                            ax.text(0.1, y_pos, f'Total messages: {len(g2_messages)}', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'NPUs involved: {sorted(g2_messages["src_npu"].unique())}', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'Time range: {g2_messages["start_time"].min():.0f} - {g2_messages["end_time"].max():.0f} ns', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'Simulation time: {(g2_messages["end_time"].max() - g2_messages["start_time"].min())/1000:.2f} μs', 
                                   fontsize=12, transform=ax.transAxes)
                            
                            # NPU statistics
                            y_pos -= 0.1
                            ax.text(0.1, y_pos, 'Per-NPU Statistics:', 
                                   fontsize=14, fontweight='bold', transform=ax.transAxes)
                            
                            npu_ids = sorted(g2_activities.keys())[:10]  # First 10 NPUs
                            for npu_id in npu_ids:
                                if y_pos < 0.1:
                                    break
                                y_pos -= 0.05
                                activities = g2_activities[npu_id]
                                if activities:
                                    total_active = sum(a['end_time'] - a['start_time'] for a in activities)
                                    span = max(a['end_time'] for a in activities) - min(a['start_time'] for a in activities)
                                    utilization = (total_active / span * 100) if span > 0 else 0
                                    max_other_active = max(a['concurrent_count'] for a in activities)
                                    
                                    ax.text(0.1, y_pos, 
                                           f'NPU {npu_id}: {len(activities)} msgs, {utilization:.1f}% util, {max_other_active} max other active', 
                                           fontsize=10, transform=ax.transAxes)
                            
                            ax.set_xlim(0, 1)
                            ax.set_ylim(0, 1)
                            ax.axis('off')
                            pdf.savefig(fig, bbox_inches='tight')
                            plt.close(fig)
                        else:
                            print(f"     ⚠️ No activities found for G2")
                    else:
                        print(f"     ⚠️ No G2 messages found")
                except Exception as e:
                    print(f"     ❌ Error processing G2: {e}")
            
            # Process NS3 if available
            if run_info['has_ns3']:
                try:
                    print(f"   📊 Generating NS3 plot...")
                    ns3_messages = parse_ns3_network_trace(run_info['ns3_trace'])
                    
                    if not ns3_messages.empty:
                        ns3_activities = extract_npu_activity(ns3_messages)
                        
                        if ns3_activities:
                            fig = generate_plot_for_pdf(
                                ns3_activities, 
                                f"NS3 Network Simulation - {run_name}",
                                time_unit='us'
                            )
                            pdf.savefig(fig, bbox_inches='tight', dpi=150)
                            plt.close(fig)
                            plot_count += 1
                            
                            # Add statistics page
                            fig, ax = plt.subplots(figsize=(11, 8.5))
                            ax.text(0.1, 0.9, f'NS3 Statistics - {run_name}', 
                                   fontsize=16, fontweight='bold', transform=ax.transAxes)
                            
                            y_pos = 0.8
                            ax.text(0.1, y_pos, f'Total messages: {len(ns3_messages)}', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'NPUs involved: {sorted(ns3_messages["src_npu"].unique())}', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'Time range: {ns3_messages["start_time"].min():.0f} - {ns3_messages["end_time"].max():.0f} ns', 
                                   fontsize=12, transform=ax.transAxes)
                            y_pos -= 0.05
                            ax.text(0.1, y_pos, f'Simulation time: {(ns3_messages["end_time"].max() - ns3_messages["start_time"].min())/1000:.2f} μs', 
                                   fontsize=12, transform=ax.transAxes)
                            
                            # NPU statistics
                            y_pos -= 0.1
                            ax.text(0.1, y_pos, 'Per-NPU Statistics:', 
                                   fontsize=14, fontweight='bold', transform=ax.transAxes)
                            
                            npu_ids = sorted(ns3_activities.keys())[:10]  # First 10 NPUs
                            for npu_id in npu_ids:
                                if y_pos < 0.1:
                                    break
                                y_pos -= 0.05
                                activities = ns3_activities[npu_id]
                                if activities:
                                    total_active = sum(a['end_time'] - a['start_time'] for a in activities)
                                    span = max(a['end_time'] for a in activities) - min(a['start_time'] for a in activities)
                                    utilization = (total_active / span * 100) if span > 0 else 0
                                    max_other_active = max(a['concurrent_count'] for a in activities)
                                    
                                    ax.text(0.1, y_pos, 
                                           f'NPU {npu_id}: {len(activities)} msgs, {utilization:.1f}% util, {max_other_active} max other active', 
                                           fontsize=10, transform=ax.transAxes)
                            
                            ax.set_xlim(0, 1)
                            ax.set_ylim(0, 1)
                            ax.axis('off')
                            pdf.savefig(fig, bbox_inches='tight')
                            plt.close(fig)
                        else:
                            print(f"     ⚠️ No activities found for NS3")
                    else:
                        print(f"     ⚠️ No NS3 messages found")
                except Exception as e:
                    print(f"     ❌ Error processing NS3: {e}")
        
        # Summary page
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.text(0.5, 0.8, 'Report Summary', 
                fontsize=20, fontweight='bold', ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.7, f'Total plots generated: {plot_count}', 
                fontsize=14, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.6, f'Runs processed: {len(all_runs)}', 
                fontsize=14, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.5, f'G2 simulations: {g2_count}', 
                fontsize=14, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.4, f'NS3 simulations: {ns3_count}', 
                fontsize=14, ha='center', transform=ax.transAxes)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    
    print(f"\\n✅ PDF report generated successfully!")
    print(f"   📄 File: {pdf_filename}")
    print(f"   📊 Total plots: {plot_count}")
    return pdf_filename

# Generate the comprehensive PDF report
if all_runs:
    pdf_file = generate_comprehensive_pdf_report()
else:
    print("❌ No simulation runs found. Please check the base_path and workload configuration.")